In [1]:
import pandas as pd
import numpy as np

In [2]:
def load_and_combine_data(file_path_1, file_path_2):
    print("Loading and combining datasets...")
    df1 = pd.read_csv(file_path_1, encoding='latin1')
    df2 = pd.read_csv(file_path_2, encoding='latin1')

    # Clean column names BEFORE combining them
    df1.rename(columns=lambda x: x.replace('ï»¿', '').strip(), inplace=True)
    df2.rename(columns=lambda x: x.replace('ï»¿', '').strip(), inplace=True)
    
    # Combine both years into a single dataframe
    df_combined = pd.concat([df1, df2], ignore_index=True)
    print(f"Combined dataset shape: {df_combined.shape[0]:,} rows.")
    return df_combined

In [3]:
def clean_retail_pipeline(df):
    print("\nStarting basic cleaning...")
    df_clean = df.copy()

    # 0. Filter by country (United Kingdom)
    df_clean = df_clean[df_clean['Country'] == 'United Kingdom']
    df_clean.drop(columns=['Country'], inplace=True)
    
    # 1. Handle Column Name Variations
    col_customer = 'Customer ID' if 'Customer ID' in df_clean.columns else 'CustomerID'
    col_price = 'Price' if 'Price' in df_clean.columns else 'UnitPrice'
    
    # 2. Drop Exact Duplicates & Missing Values
    df_clean.drop_duplicates(inplace=True)
    df_clean.dropna(subset=[col_customer, 'Description', 'Invoice'], inplace=True)
    
    # 3. Format Data Types
    df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
    # Convert Customer ID to string without the .0 float formatting
    df_clean[col_customer] = df_clean[col_customer].astype(int).astype(str)
    # Standardize string formatting for Description
    df_clean['Description'] = df_clean['Description'].astype(str).str.upper().str.strip()
    
    # 4. Filter Anomalies (Returns, Free Items, System Codes)
    df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean[col_price] > 0)]
    invalid_prefixes = ('ADJUST', 'TEST')
    df_clean = df_clean[~df_clean['StockCode'].str.startswith(invalid_prefixes)]
    invalid_codes = ['C2', 'POST', 'D', 'M', 'PADS', 'DOT', 'CRUK', 'BANK CHARGES', 'AMAZONFEE']
    df_clean = df_clean[~df_clean['StockCode'].astype(str).str.upper().isin(invalid_codes)]
    
    print("\nStarting advanced product standardization (Latest Transaction Method)...")
    
    # 5. Master Data Management: Standardize Descriptions to Latest Entry
    # Sort chronologically
    df_clean = df_clean.sort_values(by='InvoiceDate', ascending=True)
    
    # Extract the most recent description for every StockCode
    latest_descriptions = df_clean.drop_duplicates(subset=['StockCode'], keep='last')
    description_mapping = latest_descriptions.set_index('StockCode')['Description'].to_dict()
    
    # Map the latest description back to all historical transactions
    df_clean['Description'] = df_clean['StockCode'].map(description_mapping)
    
    # Remove BOM
    df_clean.rename(columns=lambda x: x.replace('ï»¿', '').strip(), inplace=True)
    # Jika ada kolom 'Invoice' kosong di ujung kanan akibat pergeseran delimiter, hapus juga:
    if 'Invoice' in df_clean.columns and df_clean.columns.duplicated().any():
        df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()]

    # Sort by date
    df_clean = df_clean.sort_values(by='InvoiceDate').reset_index(drop=True)
    
    print("\n" + "="*50)
    print("PIPELINE COMPLETE")
    print("="*50)
    print(f"Final cleaned dataset shape: {df_clean.shape[0]:,} rows.")
    
    return df_clean

In [4]:
# ==========================================
# EXECUTION BLOCK (Update paths if necessary)
# ==========================================
# Kaggle typically stores datasets in /kaggle/input/dataset-name/
path_2009 = '/kaggle/input/datasets/felixoctavius/online-retail-ii/online_retail_II_2009_2010.csv'
path_2010 = '/kaggle/input/datasets/felixoctavius/online-retail-ii/online_retail_II_2010_2011.csv'

# 1. Combine
df_raw = load_and_combine_data(path_2009, path_2010)

# 2. Clean
df_final = clean_retail_pipeline(df_raw)
   
# 3. Preview
display(df_final.head())

Loading and combining datasets...
Combined dataset shape: 1,067,371 rows.

Starting basic cleaning...

Starting advanced product standardization (Latest Transaction Method)...

PIPELINE COMPLETE
Final cleaned dataset shape: 699,610 rows.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085
4,489434,21232,STRAWBERRY CERAMIC TRINKET POT,24,2009-12-01 07:45:00,1.25,13085


In [5]:
df_final.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699610 entries, 0 to 699609
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      699610 non-null  object        
 1   StockCode    699610 non-null  object        
 2   Description  699610 non-null  object        
 3   Quantity     699610 non-null  int64         
 4   InvoiceDate  699610 non-null  datetime64[ns]
 5   Price        699610 non-null  float64       
 6   Customer ID  699610 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 175.3 MB


In [6]:
# df_final.to_csv('/kaggle/working/cleaned_uk.csv', index=False)
df_final.to_csv('/kaggle/working/cleaned_uk_v2.csv', index=False)

### Statistical details of a transactional database
The performance of a mining algorithm primarily depends on the following two key factors:

1. Distribution of items’ frequencies and
2. Distribution of transaction length

Thus, it is important to know the statistical details of a database. PAMI provides inbuilt classes and functions methods to get the statistical details of a database. In this page, we provide the details of methods to get statistical details from a transactional database.

from https://udaylab.github.io/PAMI/manuals/utilityDatabaseStats.html

In [7]:
# utk statistika dari library pami
!pip install pami==2026.1.19.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 995.1 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 7.6 MB/s eta 0:00:00
  Created wheel for JsonForm: filename=JsonForm-0.0.2-py3-none-any.whl size=3311 sha256=af39a6bc8d52496c2a2343364a7330e1885ec7659288bfd6fe287d91d0ab8787
  Stored in directory: /root/.cache/pip/wheels/0b/29/3c/f5b5085becdbee0b282b60cda0028607f67adf8f099316a4a7
  Created wheel for JsonSir: filename=JsonSir-0.0.2-py3-none-any.whl size=4754 sha256=a2b1446749a657a22d95e900ba

In [8]:
# 1. Hitung nilai utility asal (Quantity * Price)
df_final['Utility'] = df_final['Quantity'] * df_final['Price']

# 2. Pastikan StockCode berupa string
df_final['StockCode'] = df_final['StockCode'].astype(str)

pami_output_file = '/kaggle/working/online_retail_pami.txt'
sep = '\t' 

print("Memulai konversi ke format PAMI (Nilai Utility dikalikan 100)...")

with open(pami_output_file, 'w', encoding='utf-8') as f:
    for invoice, group in df_final.groupby('Invoice'):
        items = group['StockCode'].tolist()
        
        # Kalikan setiap utility dengan 100 dan ubah menjadi integer bulat
        utilities = [int(round(u * 100)) for u in group['Utility']]
        total_utility = sum(utilities)
        
        items_str = sep.join(items)
        # Ubah list integer menjadi string yang dipisahkan TAB
        utilities_str = sep.join(str(u) for u in utilities)
        
        # Tulis ke file dengan format itemA\titemB:total_utility:utilA\tutilB
        pami_line = f"{items_str}:{total_utility}:{utilities_str}\n"
        f.write(pami_line)

print(f"Konversi selesai! File integer disimpan di: {pami_output_file}")

Memulai konversi ke format PAMI (Nilai Utility dikalikan 100)...
Konversi selesai! File integer disimpan di: /kaggle/working/online_retail_pami.txt


In [9]:
import PAMI.extras.dbStats.UtilityDatabase as uds

In [10]:
# Tentukan path file hasil konversi tadi
inputFile = '/kaggle/working/online_retail_pami.txt'
        
# PAMI default menggunakan tab separator, jadi tidak perlu di-override jika memakai tab
obj = uds.UtilityDatabase(inputFile)
obj.run()

# Statistika dataset setelah cleaning
print(f'Database size : {obj.getDatabaseSize()}')
print(f'Total number of items : {obj.getTotalNumberOfItems()}')
print(f'Database sparsity : {obj.getSparsity()}')  # Diubah dari printf ke print
print(f'Minimum Transaction Size : {obj.getMinimumTransactionLength()}')
print(f'Average Transaction Size : {obj.getAverageTransactionLength()}')
print(f'Maximum Transaction Size : {obj.getMaximumTransactionLength()}')
print(f'Standard Deviation Transaction Size : {obj.getStandardDeviationTransactionLength()}')
print(f'Variance in Transaction Sizes : {obj.getVarianceTransactionLength()}') # Diperbaiki kurung & spasi
print(f'Total utility : {obj.getTotalUtility()}')
print(f'Minimum utility : {obj.getMinimumUtility()}')
print(f'Average utility : {obj.getAverageUtility()}')
print(f'Maximum utility : {obj.getMaximumUtility()}')

# Menyimpan hasil distribusi statistik ke file csv
itemFrequencies = obj.getSortedListOfItemFrequencies()
transactionLength = obj.getTransanctionalLengthDistribution()
utility = obj.getSortedUtilityValuesOfItem()

obj.save(itemFrequencies, 'itemFrequency.csv')
obj.save(transactionLength, 'transactionSize.csv')
obj.save(utility, 'utility.csv')

Database size : 33361
Total number of items : 4605
Database sparsity : 0.9954460599005757
Minimum Transaction Size : 1
Average Transaction Size : 20.970894157848985
Maximum Transaction Size : 541
Standard Deviation Transaction Size : 23.00056674319779
Variance in Transaction Sizes : 529.041928603935
Total utility : 1428877350
Minimum utility : 38
Average utility : 310288.2410423453
Maximum utility : 22796551
